# 🤖 Proyecto: Mano Robótica Controlada por Cámara
**Autora:** Valesska  
**Hardware:** ESP32 + Placa de Expansión + Controlador de Servos PCA9685  

---

## 📐 Arquitectura del Sistema

```
┌──────────────┐       ┌─────────────────────┐   I2C    ┌──────────────┐   PWM   ┌──────────────┐
│  Cámara Web  │──────▶│  Python + MediaPipe  │─Serial──▶│     ESP32    │────────▶│   PCA9685    │──── 5 Servos
│  (webcam)   │       │  detecta 5 dedos     │  USB    │  + Expansión │  SDA/SCL│  16 canales  │    Mano 3D
└──────────────┘       └─────────────────────┘         └──────────────┘         └──────────────┘
```

### Flujo de datos
1. La cámara captura la mano real
2. MediaPipe detecta los 21 puntos de la mano
3. Python calcula si cada dedo está abierto (1) o cerrado (0)
4. Envía por USB serial: `"1,0,1,1,0\n"`
5. El ESP32 recibe el string y lo parsea
6. Manda señales I2C al PCA9685
7. El PCA9685 genera PWM para cada servo
8. Los servos mueven los dedos de la mano 3D


## 🔌 Esquema de Conexión

### ESP32 → PCA9685 (I2C)

| PCA9685 | ESP32 (placa expansión) | Descripción |
|---------|------------------------|-------------|
| SDA     | GPIO 21                | Datos I2C   |
| SCL     | GPIO 22                | Reloj I2C   |
| VCC     | 3.3V                   | Lógica      |
| GND     | GND                    | Tierra común|
| V+      | **5V externa**         | Servos ⚠️   |

### PCA9685 → Servos

| Canal PCA9685 | Dedo    | Color sugerido |
|---------------|---------|----------------|
| Canal 0       | Pulgar  | Rojo           |
| Canal 1       | Índice  | Naranja        |
| Canal 2       | Medio   | Amarillo       |
| Canal 3       | Anular  | Verde          |
| Canal 4       | Meñique | Azul           |

```
⚠️  ALIMENTACIÓN DE LOS SERVOS:
    V+ del PCA9685  ──>  Fuente 5V externa (al menos 2A)
    GND externo     ──>  GND común con ESP32
    
    NO alimentes los servos desde el ESP32, los quemarás.
```

### Diagrama general

```
           USB
  PC ──────────────── ESP32 (+ Placa Expansión)
  (Python)               │
                      I2C│ SDA=GPIO21
                         │ SCL=GPIO22
                         ▼
                    ┌─────────────────┐
                    │   PCA9685       │
                    │  (I2C addr 0x40)│
                    │                 │
                    │ CH0 ────────────┼──── Servo Pulgar
                    │ CH1 ────────────┼──── Servo Índice
                    │ CH2 ────────────┼──── Servo Medio
                    │ CH3 ────────────┼──── Servo Anular
                    │ CH4 ────────────┼──── Servo Meñique
                    │                 │
                    │ V+ ─────────────┼──── 5V externo (fuente)
                    │ GND ────────────┼──── GND común
                    └─────────────────┘
```


## 📦 Librerías necesarias

### En Python (ya instaladas):
```
pip install opencv-python mediapipe pyserial
```

### En Arduino IDE:
1. Abre Arduino IDE
2. Ve a **Tools → Manage Libraries**
3. Busca e instala: **`Adafruit PWM Servo Driver Library`**
4. También instala: **`Adafruit BusIO`** (dependencia automática)
   
En el código `.ino` se incluyen:
```cpp
#include <Wire.h>                       // I2C (viene con Arduino)
#include <Adafruit_PWMServoDriver.h>    // PCA9685
```


In [ ]:
# Verificar dependencias Python
import subprocess, sys

paquetes = ['opencv-python', 'mediapipe', 'pyserial']
for p in paquetes:
    result = subprocess.run([sys.executable, '-m', 'pip', 'show', p],
                            capture_output=True, text=True)
    estado = '✅' if result.returncode == 0 else '❌'
    ver = ''
    if result.returncode == 0:
        ver = [l for l in result.stdout.split('\n') if l.startswith('Version')][0]
    print(f'{estado} {p}  {ver}')

In [ ]:
# Detectar puertos seriales disponibles
# Conecta el ESP32 y ejecuta esta celda para ver en qué COM está
import serial.tools.list_ports

puertos = list(serial.tools.list_ports.comports())
if puertos:
    print('Puertos COM detectados:')
    print()
    for p in puertos:
        print(f'  {p.device:8}  {p.description}')
    print()
    print('Busca el que diga "CP210x", "CH340" o "UART" -> ese es tu ESP32')
else:
    print('No se detectaron puertos. Verifica que el ESP32 esté conectado.')

## 🤖 Código Arduino para ESP32 + PCA9685

El archivo está en: `arduino/mano_robotica_esp32.ino`

### Pasos para subir el código:
1. Abre **Arduino IDE**
2. Ve a *File → Open* → selecciona `mano_robotica_esp32.ino`
3. En *Tools → Board* → selecciona **ESP32 Dev Module**
4. En *Tools → Port* → selecciona el COM del ESP32
5. En *Tools → Manage Libraries* → instala **Adafruit PWM Servo Driver Library**
6. Presiona **Upload** ⬆️
7. Abre el **Serial Monitor** (115200 baud) para ver los mensajes de confirmación

### Lógica del código Arduino:
```
Recibe por serial: "1,0,1,1,0\n"
                    │ │ │ │ │
                    │ │ │ │ └─ Meñique  → Canal 4: ABIERTO=150 o CERRADO=500
                    │ │ │ └─── Anular   → Canal 3
                    │ │ └───── Medio    → Canal 2
                    │ └─────── Índice   → Canal 1
                    └───────── Pulgar   → Canal 0: ABIERTO=150 o CERRADO=450

El valor de pulso del PCA9685 se convierte en PWM:
    150  →  ~0.5ms  → 0°   (servo extendido)
    375  →  ~1.5ms  → 90°  (servo centro)
    600  →  ~2.5ms  → 180° (servo doblado)
```


In [ ]:
# Ver el código Arduino
ruta_ino = r'C:\Users\vales\.gemini\antigravity\scratch\mano_robotica\arduino\mano_robotica_esp32.ino'
with open(ruta_ino, encoding='utf-8') as f:
    print(f.read())

## 🔧 Calibración de los Servos

El PCA9685 trabaja con valores de pulso de 0 a 4095 (12 bits).  
Para servos SG90 estándar:

| Valor PCA | Ángulo aprox | Posición |
|-----------|-------------|----------|
| 100       | ~0°         | Mínimo   |
| 150       | ~0°         | Abierto  |
| 375       | ~90°        | Centro   |
| 500       | ~150°       | Cerrado  |
| 600       | ~180°       | Máximo   |

Ajusta estos valores en el `.ino` según tu mano impresa:
```cpp
const int ABIERTO[5] = { 150, 150, 150, 150, 150 };
const int CERRADO[5] = { 450, 500, 500, 500, 500 };
```


In [ ]:
# ═══════════════════════════════════════════════════════════
#  CELDA DE CALIBRACIÓN
#  Conecta el ESP32, ajusta PUERTO y ejecuta
# ═══════════════════════════════════════════════════════════
import serial, time

PUERTO = 'COM4'   # <- cambia al COM de tu ESP32

try:
    esp = serial.Serial(PUERTO, 9600, timeout=2)
    time.sleep(2)
    print(f'Conectado al ESP32 en {PUERTO}\n')

    secuencia = [
        ('Todos ABIERTOS           ', '1,1,1,1,1'),
        ('Todos CERRADOS (puno)    ', '0,0,0,0,0'),
        ('Solo PULGAR abierto     ', '1,0,0,0,0'),
        ('Solo INDICE abierto     ', '0,1,0,0,0'),
        ('Solo MEDIO abierto      ', '0,0,1,0,0'),
        ('Solo ANULAR abierto     ', '0,0,0,1,0'),
        ('Solo MENIQUE abierto    ', '0,0,0,0,1'),
        ('Numero UNO  (indice)    ', '0,1,0,0,0'),
        ('Numero DOS              ', '0,1,1,0,0'),
        ('Numero TRES             ', '0,1,1,1,0'),
        ('Numero CUATRO           ', '0,1,1,1,1'),
        ('Numero CINCO            ', '1,1,1,1,1'),
        ('LIKE (pulgar arriba)    ', '1,0,0,0,0'),
        ('ROCK (indice+menique)   ', '0,1,0,0,1'),
        ('Volver a ABIERTO        ', '1,1,1,1,1'),
    ]

    for nombre, cmd in secuencia:
        print(f'  [{cmd}]  {nombre}')
        esp.write((cmd + '\n').encode())
        # Leer respuesta del ESP32 si la hay
        time.sleep(0.1)
        if esp.in_waiting:
            print('  ESP32:', esp.readline().decode().strip())
        time.sleep(1.5)

    esp.close()
    print('\nCalibración completa.')

except Exception as e:
    print(f'Error de conexión: {e}')
    print('Verifica que el ESP32 esté conectado y el puerto sea correcto.')

## 🚀 Ejecutar el Sistema Completo

### Checklist antes de arrancar:
- [ ] Código `.ino` subido al ESP32
- [ ] Servos conectados a los canales 0-4 del PCA9685
- [ ] Fuente 5V externa conectada al V+ del PCA9685
- [ ] ESP32 conectado por USB al PC
- [ ] Puerto COM identificado (celda de detección arriba)

### Activar el serial en el script Python:
Edita el archivo `mano_robotica.py` y cambia:
```python
USAR_SERIAL   = True    # <- activar
PUERTO_SERIAL = 'COM4'  # <- puerto de tu ESP32
```

Luego ejecuta:
```
python mano_robotica.py
```


In [ ]:
# Lanzar el sistema completo desde el notebook
import subprocess, sys

script = r'C:\Users\vales\.gemini\antigravity\scratch\mano_robotica\mano_robotica.py'
print('Iniciando... (presiona q en la ventana para salir)')
subprocess.run([sys.executable, script])

## 📊 Tabla de Gestos

| Gesto           | Pulgar | Índice | Medio | Anular | Meñique | Serial         |
|-----------------|--------|--------|-------|--------|---------|----------------|
| ✋ Mano abierta | 1      | 1      | 1     | 1      | 1       | `1,1,1,1,1`    |
| ✊ Puño         | 0      | 0      | 0     | 0      | 0       | `0,0,0,0,0`    |
| ☝️ Número 1    | 0      | 1      | 0     | 0      | 0       | `0,1,0,0,0`    |
| ✌ Número 2     | 0      | 1      | 1     | 0      | 0       | `0,1,1,0,0`    |
| 3 dedos         | 0      | 1      | 1     | 1      | 0       | `0,1,1,1,0`    |
| 👍 Like         | 1      | 0      | 0     | 0      | 0       | `1,0,0,0,0`    |
| 🤘 Rock         | 0      | 1      | 0     | 0      | 1       | `0,1,0,0,1`    |

---

## ✅ Requisitos del Proyecto (checklist del profe)

| Requisito | Implementación | Estado |
|-----------|---------------|--------|
| Conectar microcontrolador a servomotores | ESP32 → PCA9685 → 5 servos vía I2C/PWM | ✅ |
| Conectar sensor al microcontrolador | Webcam → Python → Serial → ESP32 | ✅ |
| Programar microcontrolador para interpretar gestos | `.ino` parsea serial y controla PCA9685 | ✅ |
| Prueba y calibración de movimientos | Celda de calibración con secuencia de gestos | ✅ |
